In [0]:
%sql
-- The PVDAQ Public Data Lake solar PV data are hosted on Amazon S3 at 's3://oedi-data-lake/pvdaq/parquet/'. 

-- Within this /parquet/ directory are eight subdirectories that correspond to metadata lookup tables: inverters, meters, metrics, mount, other_instruments, site, system

-- Also within /parquet/ is a subdirectory, pvdata, that contains the time series data itself, partitioned by system_id, year, month, and day.

CREATE CATALOG IF NOT EXISTS pvdaq_catalog;
CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.bronze;

USE CATALOG pvdaq_catalog;
USE SCHEMA bronze;
SELECT current_catalog(), current_schema();

-- Extracting and loading the eight metadata/lookup tables, manually selecting the table schemas from the PVDAQ documentation

CREATE TABLE IF NOT EXISTS inverters AS SELECT 
  inverter_id,
  name,
  manufacturer,
  model,
  serial_num,
  num_strings,
  modules_per_string,
  type,
  quantity,
  time_interval,
  site_id,
  system_id,
  comments
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/inverters/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS meters AS SELECT 
  meter_id,
  name,
  manufacturer,
  model,
  serial_num,
  time_interval,
  type,
  site_id,
  system_id,
  comments
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/meters/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS metrics AS SELECT 
  system_id,
  metric_id,
  sensor_name,
  common_name,
  raw_units,
  units,
  calc_scale,
  calc_offset,
  calc_details,
  aggregation_type,
  source_type,
  source_id,
  comments,
  standard_name
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/metrics/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS modules AS SELECT 
  module_id,
  name,
  inverter_id,
  manufacturer,
  model,
  serial_num,
  type,
  quantity,
  reference_module,
  start_on,
  end_on,
  site_id,
  system_id,
  comments
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/modules/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS mount AS SELECT
  mount_id,
  name,
  manufacturer,
  model,
  azimuth,
  tilt,
  tracking,
  type,
  site_id,
  system_id
  -- schema documentation drift: comments not present in mount table
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/mount/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS other_instruments AS SELECT 
  instrument_id,
  name,
  manufacturer,
  model,
  serial_num,
  time_interval,
  type,
  site_id,
  system_id,
  comments
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/other-instruments/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS site AS SELECT 
  site_id,
  system_id,
  public_name,
  location,
  latitude,
  longitude,
  elevation,
  av_pressure,
  av_temp,
  climate_type
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/site/',
    format => 'parquet'
  );

CREATE TABLE IF NOT EXISTS system AS SELECT 
  system_id,
  site_id,
  public_name,
  area,
  power,
  started_on,
  ended_on,
  comments
FROM
  read_files(
    's3://oedi-data-lake/pvdaq/parquet/system/',
    format => 'parquet'
  );


In [0]:
%sql

-- This notebook cell explores the size of the total dataset and selects a sample to do analysis on

-- Estimating the size of the time series dataset (/pvdata/)
-- This is a deeply partitioned S3 dataset, partitioned into /pvdata/system_id/year/month/day, with rows typically at a 15 minute resolution per day

SELECT COUNT(*) AS single_month_rows
FROM read_files(
  's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id=10/year=2020/month=1/day=23',
  format => 'parquet'
);
-- 10,640 rows for a single day

SELECT COUNT(*) 
  FROM system;
-- 157 systems in total

-- With 157 systems, and several years of data for each system, there are likely multiple billions of rows in this dataset

-- Pulling only systems that have data present in 2020
SELECT DISTINCT system_id
FROM read_files(
  's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id=*/year=2020/*/*/*.parquet',
  format => 'parquet'
)
ORDER BY CAST(system_id AS INT);
-- Systems that have data in 2020: 2, 3, 4, 10, 33, 34, 35, 50, 51, 1199, 1200, 1201, 1202, 1203, 1208, 1239, 1276, 1277, 1278, 1283, 1289, 1332, 1367, 1368, 1369, 1403, 1418, 1419, 1420, 1423, 2045

-- Randomly sampling 5 systems from the 2020 systems to provide a snapshot analysis of the performance of the entire fleet in 2020
WITH systems_2020 AS (
  SELECT DISTINCT system_id
  FROM read_files(
    's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id=*/year=2020/*/*/*.parquet',
    format => 'parquet'
  )
)
SELECT system_id
  FROM systems_2020
  ORDER BY RAND()
  LIMIT 5;
-- systems sampled: 1332, 1276, 1239, 1278, 3

-- Calculating total number of rows for the sampled systems in 2020
SELECT 
  system_id,
  COUNT(*) AS total_system_rows
FROM read_files(
  's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id={1332,1276,1239,1278,3}/year=2020/*/*/*.parquet',
  format => 'parquet'
)
GROUP BY system_id
ORDER BY CAST(system_id AS INT);
-- Rows per sampled system:
-- 3: 2,974,576
-- 1239: 159,710
-- 1276: 281,257
-- 1278: 457,298
-- 1332: 26,301,960
-- Total rows: 30,174,801



In [0]:
%sql
-- Ingesting the 2020 from the five system sample into Delta Lake

USE CATALOG pvdaq_catalog;
USE SCHEMA bronze;

-- Creating the Bronze table
CREATE TABLE IF NOT EXISTS pvdata_2020_sample (
  system_id INT,
  year INT,
  month INT,
  day INT,
  measured_on TIMESTAMP,
  utc_measured_on TIMESTAMP,
  metric_id INT,
  value DOUBLE
);

-- Running idempotent COPY INTO with metadata path extraction
-- I needed to use regular expressions to extract the system_id values out of the partitioned parquet files.
-- This will allow later joining on the metadata tables.
COPY INTO pvdata_2020_sample
FROM (
  SELECT 
    CAST(regexp_extract(_metadata.file_path, 'system_id=([0-9]+)', 1) AS INT) AS system_id,
    CAST(regexp_extract(_metadata.file_path, 'year=([0-9]+)', 1) AS INT) AS year,
    CAST(regexp_extract(_metadata.file_path, 'month=([0-9]+)', 1) AS INT) AS month,
    CAST(regexp_extract(_metadata.file_path, 'day=([0-9]+)', 1) AS INT) AS day,
    measured_on,
    utc_measured_on,
    metric_id,
    value
  FROM 's3a://oedi-data-lake/pvdaq/parquet/pvdata/system_id={1332,1276,1239,1278,3}/year=2020/*/*/*.parquet'
)
FILEFORMAT = PARQUET;

-- Check table size
SELECT COUNT(*)
FROM pvdaq_catalog.bronze.pvdata_2020_sample;


